In [ ]:
# Data Engineering Practicals
   #   Practical-2

In [ ]:
# Name: Insiya Shoeb Bobde
# Rollno: 06
# Student-id: 5115304

### Step 1: Install `pyodbc`

`pyodbc` is a Python library that allows you to connect to and use various databases, including SQL Server, via ODBC. First, you need to install it.

In [ ]:
pip install pyodbc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.3/340.3 kB 19.2 MB/s eta 0:00:00


### Step 2: Establish a Connection to SQL Server

To connect to a SQL Server database, you'll need connection details such as the server address, database name, username, and password. You also need an ODBC driver installed on your system where Python is running. For Colab, you might need to install an appropriate driver or use a cloud-hosted SQL Server that's accessible.

**Note:** For this example, I'll provide a general structure. You'll need to replace the placeholder values (`<SERVER_NAME>`, `<DATABASE_NAME>`, `<USERNAME>`, `<PASSWORD>`) with your actual SQL Server credentials and ensure the correct ODBC Driver is specified. If you are running this in a local environment, you would typically have a driver like `ODBC Driver 17 for SQL Server`.

In [ ]:
import sqlite3
import os

# Define the database file name
db_file = 'BI_ETL_Practical2.db'

cnxn = None
try:
    # Connect to SQLite database (creates the file if it doesn't exist)
    cnxn = sqlite3.connect(db_file)
    cursor = cnxn.cursor()
    print(f"Successfully connected to SQLite database: {db_file}")

    # Optionally, clear existing tables for a fresh run if the file already exists
    # (Uncomment if you want to always start with a clean slate)
    # cursor.execute("DROP TABLE IF EXISTS Processed_Product_Data;")
    # cursor.execute("DROP TABLE IF EXISTS Raw_Product_Data;")
    # cnxn.commit()
    # print("Existing tables dropped (if any).")

except sqlite3.Error as ex:
    print(f"Error connecting to SQLite database: {ex}")
    cnxn = None

# Ensure connection is established before proceeding
if cnxn is None:
    raise Exception("Failed to establish database connection.")

Successfully connected to SQLite database: BI_ETL_Practical2.db


### Step 3: Execute your SQL ETL Process

Now you can execute the SQL commands from your previous cell. We'll wrap them in Python strings and use the `cursor.execute()` method. Note that `GO` statements are specific to SQL Server management tools and should be removed when executing SQL via `pyodbc`.

In [ ]:
if cnxn:
    cursor = cnxn.cursor()
    try:
        # Step 2: Create Staging Table
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS Raw_Product_Data
        (
            ProductID INTEGER,
            ProductName VARCHAR(100),
            Category VARCHAR(50),
            Quantity INTEGER,
            Price DECIMAL(10,2),
            SaleDate VARCHAR(20),
            City VARCHAR(50)
        );
        """)
        print("Raw_Product_Data table created or already exists.")

        # Step 3: Extract / Insert Raw Data
        # Check if data already exists to prevent duplicate inserts on re-run
        cursor.execute("SELECT COUNT(*) FROM Raw_Product_Data;")
        if cursor.fetchone()[0] == 0:
            cursor.execute("""
            INSERT INTO Raw_Product_Data
            (ProductID, ProductName, Category, Quantity, Price, SaleDate, City)
            VALUES
            (101, 'Laptop', 'Electronics', 2, 55000, '2026-01-10', 'Mumbai'),
            (102, 'Mobile', 'Electronics', 3, 25000, '2026-01-12', 'Pune'),
            (103, 'Chair', 'Furniture', 5, 4500, '2026-01-15', 'Nashik'),
            (104, 'Table', 'Furniture', 2, 8000, '2026-01-18', 'Mumbai'),
            (105, 'Headphones', 'Electronics', 4, 2000, '2026-01-20', 'Nagpur');
            """)
            print("Data inserted into Raw_Product_Data.")
        else:
            print("Raw_Product_Data already contains data. Skipping insert.")

        # Step 4: Display Raw Data (using fetchall() and pandas for better display)
        cursor.execute("SELECT * FROM Raw_Product_Data;")
        raw_data = cursor.fetchall()
        import pandas as pd
        raw_df = pd.DataFrame.from_records(raw_data, columns=[desc[0] for desc in cursor.description])
        print("\nRaw Product Data:")
        display(raw_df)

        # Step 5: Create Final Table
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS Processed_Product_Data
        (
            ProductID INTEGER PRIMARY KEY,
            ProductName VARCHAR(100),
            Category VARCHAR(50),
            Quantity INTEGER,
            Price DECIMAL(10,2),
            SaleDate DATE,
            City VARCHAR(50),
            TotalAmount DECIMAL(12,2)
        );
        """)
        print("Processed_Product_Data table created or already exists.")

        # Step 6: Transform and Load Data
        # The date is converted from VARCHAR to DATE, and the total amount is calculated.
        # Check if data already exists to prevent duplicate inserts on re-run
        cursor.execute("SELECT COUNT(*) FROM Processed_Product_Data;")
        if cursor.fetchone()[0] == 0:
            cursor.execute("""
            INSERT INTO Processed_Product_Data
            (
                ProductID,
                ProductName,
                Category,
                Quantity,
                Price,
                SaleDate,
                City,
                TotalAmount
            )
            SELECT
                ProductID,
                ProductName,
                Category,
                Quantity,
                Price,
                DATE(SaleDate), -- Changed CONVERT(DATE, SaleDate) to DATE(SaleDate) for SQLite
                City,
                Quantity * Price
            FROM Raw_Product_Data;
            """)
            print("Data transformed and loaded into Processed_Product_Data.")
        else:
            print("Processed_Product_Data already contains data. Skipping transform and load.")

        # Step 7: Display Final Transformed Data
        cursor.execute("SELECT * FROM Processed_Product_Data;")
        processed_data = cursor.fetchall()
        processed_df = pd.DataFrame.from_records(processed_data, columns=[desc[0] for desc in cursor.description])
        print("\nProcessed Product Data:")
        display(processed_df)

        # Step 8: Perform Analysis
        print("\nPerforming Analysis:")
        # Total Sales
        cursor.execute("SELECT SUM(TotalAmount) AS Total_Sales FROM Processed_Product_Data;")
        total_sales = cursor.fetchone()[0]
        print(f"Total Sales: {total_sales}")

        # Category-wise Sales
        cursor.execute("""
        SELECT
            Category,
            SUM(TotalAmount) AS Category_Total
        FROM Processed_Product_Data
        GROUP BY Category;
        """)
        category_sales = cursor.fetchall()
        category_sales_df = pd.DataFrame.from_records(category_sales, columns=[desc[0] for desc in cursor.description])
        print("\nCategory-wise Sales:")
        display(category_sales_df)

        # City-wise Sales
        cursor.execute("""
        SELECT
            City,
            SUM(TotalAmount) AS City_Total
        FROM Processed_Product_Data
        GROUP BY City;
        """)
        city_sales = cursor.fetchall()
        city_sales_df = pd.DataFrame.from_records(city_sales, columns=[desc[0] for desc in cursor.description])
        print("\nCity-wise Sales:")
        display(city_sales_df)

        cnxn.commit() # Commit all transactions
        print("All SQL operations completed successfully and committed.")

    except sqlite3.Error as ex:
        print(f"Error during SQL execution: {ex}")
        cnxn.rollback() # Rollback in case of error
        print("Transaction rolled back.")
    finally:
        cursor.close()
        cnxn.close()
        print("Database connection closed.")
else:
    print("No active database connection to execute SQL queries.")

Raw_Product_Data table created or already exists.
Data inserted into Raw_Product_Data.

Raw Product Data:


,ProductID,ProductName,Category,Quantity,Price,SaleDate,City
0,101,Laptop,Electronics,2,55000,2026-01-10,Mumbai
1,102,Mobile,Electronics,3,25000,2026-01-12,Pune
2,103,Chair,Furniture,5,4500,2026-01-15,Nashik
3,104,Table,Furniture,2,8000,2026-01-18,Mumbai
4,105,Headphones,Electronics,4,2000,2026-01-20,Nagpur


Processed_Product_Data table created or already exists.
Data transformed and loaded into Processed_Product_Data.

Processed Product Data:


,ProductID,ProductName,Category,Quantity,Price,SaleDate,City,TotalAmount
0,101,Laptop,Electronics,2,55000,2026-01-10,Mumbai,110000
1,102,Mobile,Electronics,3,25000,2026-01-12,Pune,75000
2,103,Chair,Furniture,5,4500,2026-01-15,Nashik,22500
3,104,Table,Furniture,2,8000,2026-01-18,Mumbai,16000
4,105,Headphones,Electronics,4,2000,2026-01-20,Nagpur,8000



Performing Analysis:
Total Sales: 231500

Category-wise Sales:


,Category,Category_Total
0,Electronics,193000
1,Furniture,38500



City-wise Sales:


,City,City_Total
0,Mumbai,126000
1,Nagpur,8000
2,Nashik,22500
3,Pune,75000


All SQL operations completed successfully and committed.
Database connection closed.
